## SUB Data Format

SUB data comes from the Staats- und Universitätsbibliothek Hamburg and follows METS/Alto format with specific characteristics:
- Directory structure: `[base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/`
- METS file: `PPN{ppn}_{date}.xml` (contains metadata and structure)
- Alto XML files: `{number}.xml` (e.g., `00000001.xml`, `00000002.xml`, etc.)
- PPN identifiers from GBV catalog system
- Edition names like "Ausgabe", "A1-Abendausgabe", "A2-Morgenausgabe" mapped to a, a, b

## 1. Setup Paths and Configuration

In [16]:
import sys
import os

# Set project root as PYTHONPATH for notebook
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("PYTHONPATH set for notebook:", project_root)

PYTHONPATH set for notebook: /Users/maslionok/Documents/JOB/test-epfl/impresso-text-acquisition


In [17]:
import os
import json
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
import bs4
from bs4 import BeautifulSoup

from text_preparation.importers.sub.detect import SubIssueDir, dir2issue, detect_issues, select_issues
from text_preparation.importers.sub.classes import SubNewspaperPage, SubNewspaperIssue
from PIL import Image, ImageDraw, ImageFont
from text_preparation.utils import draw_box_on_img, coords_to_xywh, coords_to_xy, rescale_coords
from text_preparation.importers.mets_alto.alto import distill_coordinates
from impresso_essentials.utils import SourceType, SourceMedium, timestamp

# SUB-specific constants
IIIF_ENDPOINT_URI = "https://iiif.sub.uni-hamburg.de/object/"
IIIF_MANIFEST_SUFFIX = "/manifest"

In [18]:
# Base directory paths
sub_source_data_dir = "../text_preparation/data/sample_data/SUB"
sub_sample_dir = "../text_preparation/data/sample_data/SUB"

# Sample newspaper alias for testing
test_alias = "hamb_echo"

# SUB-specific constants
SUB_IMG_TYPE = "illustration"
SUB_AD_TYPE = "advert"
SUB_CAPTION_TYPE = "caption"

print(f"SUB sample directory: {sub_sample_dir}")
print(f"Test newspaper: {test_alias}")

SUB sample directory: ../text_preparation/data/sample_data/SUB
Test newspaper: hamb_echo


## 2. Explore SUB Data Directory Structure

In [19]:
# Explore the SUB data directory structure
test_data_path = os.path.join(sub_sample_dir, test_alias)

print(f"Contents of {test_data_path}:")
for root, dirs, files in os.walk(test_data_path):
    level = root.replace(test_data_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files per directory
        print(f'{sub_indent}{file}')
    if len(files) > 5:
        print(f'{sub_indent}... and {len(files) - 5} more files')

Contents of ../text_preparation/data/sample_data/SUB/hamb_echo:
hamb_echo/
  1932/
    09/
      01/
        Ausgabe/
          00000010.xml
          00000004.xml
          PPN1754726119_19320901.xml
          00000005.xml
          00000007.xml
          ... and 16 more files
  1923/
    05/
      24/
        A1-Abendausgabe/
          00000004.xml
          00000002.xml
          00000003.xml
          00000001.xml
          00000003.tif
          ... and 4 more files
  1888/
    02/
      02/
        Ausgabe/
          00000004.xml
          00000005.xml
          00000007.xml
          00000006.xml
          00000002.xml
          ... and 12 more files


### Inspect METS and Alto Files

In [20]:
# Find a sample METS file and inspect structure
sample_issue_path = os.path.join(test_data_path, "1888/02/02/Ausgabe")
mets_files = [f for f in os.listdir(sample_issue_path) if f.startswith('PPN') and f.endswith('.xml')]
alto_files = [f for f in os.listdir(sample_issue_path) if f.startswith('0') and f.endswith('.xml')]

print(f"METS files: {mets_files}")
print(f"Alto files ({len(alto_files)} total): {alto_files[:3]}...")

# Load and inspect METS structure
if mets_files:
    mets_path = os.path.join(sample_issue_path, mets_files[0])
    with open(mets_path, 'r', encoding='utf-8') as f:
        mets_content = f.read()
    
    mets_soup = BeautifulSoup(mets_content, 'xml')
    
    # Extract key information
    ppn = mets_soup.find('recordIdentifier', {'source': 'gbv'})
    title = mets_soup.find('title')
    
    print(f"\nMETS Info:")
    print(f"  PPN: {ppn.text if ppn else 'Not found'}")
    print(f"  Title: {title.text if title else 'Not found'}")

METS files: ['PPN1754726119_18880202.xml']
Alto files (8 total): ['00000004.xml', '00000005.xml', '00000007.xml']...

METS Info:
  PPN: PPN1754726119
  Title: Hamburger Echo


## 3. Detect and Select Issues

In [21]:
# Test detect_issues function
detected = detect_issues(sub_source_data_dir)

print(f"Detected {len(detected)} issues in total")

for issue in detected[:5]:
    print(f"  {issue.alias} - {issue.date} - edition:{issue.edition} - {issue.path}")

Detected 3 issues in total
  hamb_echo - 1888-02-02 - edition:a - ../text_preparation/data/sample_data/SUB/hamb_echo/1888/02/02/Ausgabe
  hamb_echo - 1923-05-24 - edition:a - ../text_preparation/data/sample_data/SUB/hamb_echo/1923/05/24/A1-Abendausgabe
  hamb_echo - 1932-09-01 - edition:a - ../text_preparation/data/sample_data/SUB/hamb_echo/1932/09/01/Ausgabe


In [22]:
config_sanity_check = {
    "titles": {
        "hamb_echo": []
    },
    "exclude_titles": [],
    "year_only": False
}

selected = select_issues(sub_source_data_dir, config=config_sanity_check)

print(f"Selected {len(selected)} issues in total")

for issue in selected:
    print(f"  {issue.alias} - {issue.date} - edition:{issue.edition}")

Selected 3 issues in total
  hamb_echo - 1888-02-02 - edition:a
  hamb_echo - 1923-05-24 - edition:a
  hamb_echo - 1932-09-01 - edition:a


In [23]:
config_test_2 = {
    "titles": {
        "hamb_echo": []
    },
    "exclude_titles": [],
    "year_only": False
}

selected = select_issues(sub_source_data_dir, config=config_test_2)

print(f"Selected {len(selected)} issues in total")

for issue in selected:
    print(f"  {issue.alias} - {issue.date} - edition:{issue.edition}")

Selected 3 issues in total
  hamb_echo - 1888-02-02 - edition:a
  hamb_echo - 1923-05-24 - edition:a
  hamb_echo - 1932-09-01 - edition:a


## 4. Sanity Check: Process Selected Issues

In [24]:
sc_issues = [SubNewspaperIssue(i) for i in selected]

sc_pages = {}
for i in sc_issues:
    sc_pages[i] = []
    for p in i.pages:
        p.add_issue(i)
        p.parse()
        print(f"Format of p is {type(p)}")
        sc_pages[i].append(p)

print(f"Processed {len(sc_issues)} issues")
print(f"Total pages: {sum(len(pages) for pages in sc_pages.values())}")

Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_preparation.importers.sub.classes.SubNewspaperPage'>
Format of p is <class 'text_

In [25]:
# Display SUB issue structure
print("SUB Issue Structure:")
print(json.dumps({k: type(v).__name__ for k, v in sc_issues[0].issue_data.items()}, indent=2))

print(f"\nSUB-specific characteristics:")
print(f"  - Provider: SUB (Staats- und Universitätsbibliothek Hamburg)")
print(f"  - Identifier type: PPN (e.g., {sc_issues[0].ppn})")
print(f"  - PPN with date: {sc_issues[0].ppn_with_date}")
print(f"  - Title: {sc_issues[0].title}")
print(f"  - IIIF manifest: {sc_issues[0].issue_data.get('iiif_manifest_uri')}")
print(f"  - Edition mapping: A1→a, A2→b, A3→c, Ausgabe→a")

SUB Issue Structure:
{
  "id": "str",
  "cdt": "str",
  "ts": "str",
  "st": "str",
  "sm": "str",
  "olr": "bool",
  "i": "list",
  "pp": "list",
  "iiif_manifest_uri": "str",
  "n": "list",
  "t": "str"
}

SUB-specific characteristics:
  - Provider: SUB (Staats- und Universitätsbibliothek Hamburg)
  - Identifier type: PPN (e.g., PPN1754726119)
  - PPN with date: PPN1754726119_18880202
  - Title: Hamburger Echo
  - IIIF manifest: https://iiif.sub.uni-hamburg.de/object/PPN1754726119_18880202/manifest
  - Edition mapping: A1→a, A2→b, A3→c, Ausgabe→a


## 5. Generate and Validate Canonical Output

In [26]:
from text_preparation.utils import validate_issue_schema
# Save canonical JSON files
output_dir = "output_issues"
os.makedirs(output_dir, exist_ok=True)

for issue in sc_issues:
    output_file = os.path.join(output_dir, f"{issue.id}.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        validate_issue_schema(issue.issue_data)
        json.dump(issue.issue_data, f, ensure_ascii=False, indent=4)
    print(f"Saved: {issue.id}.json")
    
print(f"\nSaved {len(sc_issues)} canonical JSON files to {output_dir}")

Saved: hamb_echo-1888-02-02-a.json


Saved: hamb_echo-1923-05-24-a.json
Saved: hamb_echo-1932-09-01-a.json

Saved 3 canonical JSON files to output_issues


In [27]:
# Validate canonical format
def validate_canonical_format(issue):
    """Validate that issue data contains all required fields."""
    required_issue_fields = ['id', 'cdt', 'ts', 'st', 'sm', 'olr', 'i', 'pp', 'n']
    print(f"\nValidation for {issue.id}:")
    for field in required_issue_fields:
        present = field in issue.issue_data
        print(f"  {field}: {'✓' if present else '✗'}")
    
    print(f"  Pages: {len(issue.issue_data.get('pp', []))}")
    print(f"  Content items: {len(issue.issue_data.get('i', []))}")
    
    return all(field in issue.issue_data for field in required_issue_fields)

# Validate first issue
validate_canonical_format(sc_issues[0])


Validation for hamb_echo-1888-02-02-a:
  id: ✓
  cdt: ✓
  ts: ✓
  st: ✓
  sm: ✓
  olr: ✓
  i: ✓
  pp: ✓
  n: ✓
  Pages: 8
  Content items: 13


True

## 6. Deep Dive: Inspect a Single Issue's Page Data

In [28]:
test_issue = selected[0]
test_issue

IssueDirectory(provider='SUB', alias='hamb_echo', date=datetime.date(1888, 2, 2), edition='a', path='../text_preparation/data/sample_data/SUB/hamb_echo/1888/02/02/Ausgabe')

In [29]:
sub_issue = SubNewspaperIssue(test_issue)
sub_issue

In [30]:
sub_issue.issue_data

{'id': 'hamb_echo-1888-02-02-a',
 'cdt': '2025-12-02 11:41:05',
 'ts': '2025-12-02T10:41:05Z',
 'st': 'newspaper',
 'sm': 'print',
 'olr': False,
 'i': [{'m': {'id': 'hamb_echo-1888-02-02-a-i0001',
    'tp': 'page',
    'l': 'de',
    'pp': [1]},
   'l': {'id': 'PPN1754726119-PPN1754726119-00000001.xml',
    'parts': [{'comp_id': 'Page1_Block11',
      'comp_role': 'body',
      'comp_fileid': '00000001.xml',
      'comp_page_no': 1},
     {'comp_id': 'Page1_Block12',
      'comp_role': 'body',
      'comp_fileid': '00000001.xml',
      'comp_page_no': 1},
     {'comp_id': 'Page1_Block14',
      'comp_role': 'body',
      'comp_fileid': '00000001.xml',
      'comp_page_no': 1},
     {'comp_id': 'Page1_Block15',
      'comp_role': 'body',
      'comp_fileid': '00000001.xml',
      'comp_page_no': 1},
     {'comp_id': 'Page1_Block16',
      'comp_role': 'body',
      'comp_fileid': '00000001.xml',
      'comp_page_no': 1},
     {'comp_id': 'Page1_Block17',
      'comp_role': 'body',
    

In [31]:
os.listdir(test_issue.path)

['00000004.xml',
 '00000005.xml',
 '00000007.xml',
 '00000006.xml',
 '00000002.xml',
 '00000003.xml',
 '00000001.xml',
 '00000003.tif',
 '00000002.tif',
 '00000001.tif',
 '00000005.tif',
 '00000004.tif',
 '00000006.tif',
 '00000007.tif',
 'PPN1754726119_18880202.xml',
 '00000008.tif',
 '00000008.xml']

In [32]:
# Inspect pages
# Note: Page data (fw, fh) is available, but regions ('r') are empty 
# because page.parse() hasn't been called on this *specific instance*.
# We will use the `sc_issues` from the sanity check for region debugging.
print(f"Pages in issue:")
for page in sub_issue.pages:
    print(f"  Page {page.number}: {page.id} - Size: {page.page_data.get('fw')}x{page.page_data.get('fh')}")

Pages in issue:
  Page 1: hamb_echo-1888-02-02-a-p0001 - Size: 3150x4743
  Page 2: hamb_echo-1888-02-02-a-p0002 - Size: 3255x4738
  Page 3: hamb_echo-1888-02-02-a-p0003 - Size: 3150x4743
  Page 4: hamb_echo-1888-02-02-a-p0004 - Size: 3255x4738
  Page 5: hamb_echo-1888-02-02-a-p0005 - Size: 3150x4743
  Page 6: hamb_echo-1888-02-02-a-p0006 - Size: 3255x4738
  Page 7: hamb_echo-1888-02-02-a-p0007 - Size: 3150x4743
  Page 8: hamb_echo-1888-02-02-a-p0008 - Size: 3255x4738


In [33]:
# Inspect first page data from the *already processed* sanity check issues
first_page = sc_issues[0].pages[0]
print(f"First page ID: {first_page.id}")
print(f"Number of regions: {len(first_page.page_data.get('r', []))}")
print(f"\nFirst region:")
if first_page.page_data.get('r'):
    first_region = first_page.page_data['r'][0]
    print(json.dumps(first_region, indent=2))

First page ID: hamb_echo-1888-02-02-a-p0001
Number of regions: 34

First region:
{
  "c": [
    229,
    413,
    2286,
    392
  ],
  "p": [
    {
      "c": [
        229,
        413,
        2286,
        392
      ],
      "l": [
        {
          "c": [
            868,
            428,
            1050,
            30
          ],
          "t": [
            {
              "c": [
                868,
                433,
                41,
                25
              ],
              "tx": ".Ai"
            },
            {
              "c": [
                1888,
                435,
                30,
                19
              ],
              "tx": "v"
            }
          ]
        },
        {
          "c": [
            291,
            432,
            1756,
            72
          ],
          "t": [
            {
              "c": [
                291,
                449,
                50,
                55
              ],
              "

## 7. Debugging Page Regions and XML Structure

### Inspecting the ALTO XML for Page 1

In [34]:
# Get the raw XML for page 1 of the first sanity-checked issue
page_1_xml = sc_issues[0].pages[0].xml

pg_1_printspace = page_1_xml.find("PrintSpace")
pg_1_printspace

<PrintSpace HEIGHT="4114" HPOS="53" VPOS="412" WIDTH="3094">
<TextBlock HEIGHT="392" HPOS="229" ID="Page1_Block11" STYLEREFS="StyleId-13398420-38F0-4F65-A641-1EDF227E7A8A-" VPOS="413" WIDTH="2286" language="de"><Shape><Polygon POINTS="2339,413 2513,413 2513,514 2514,514 2514,732 2515,732 2515,795 2340,795 2340,796 2122,796 2122,797 1904,797 1904,798 1685,798 1685,799 1467,799 1467,800 1248,800 1248,801 1030,801 1030,802 811,802 811,803 593,803 593,804 374,804 374,805 231,805 231,734 230,734 230,515 229,515 229,423 373,423 373,422 591,422 591,421 810,421 810,420 1028,420 1028,419 1246,419 1246,418 1465,418 1465,417 1683,417 1683,416 1902,416 1902,415 2120,415 2120,414 2339,414 2339,413"/></Shape>
<TextLine HEIGHT="30" HPOS="868" STYLEREFS="StyleId-BF4EB871-825A-4935-A9A2-9EF6B42EF6BD-" VPOS="428" WIDTH="1050"><String CONTENT=".Ai" HEIGHT="25" HPOS="868" VPOS="433" WIDTH="41"/><SP HPOS="910" VPOS="429" WIDTH="977"/><String CONTENT="v" HEIGHT="19" HPOS="1888" VPOS="435" WIDTH="30"/></Text

In [35]:
# Display the parsed page_data for that same page
# This confirms the 'r' (regions) list is empty.
sc_issues[0].pages[0].page_data

{'id': 'hamb_echo-1888-02-02-a-p0001',
 'cdt': '2025-12-02 11:41:02',
 'ts': '2025-12-02T10:41:02Z',
 'st': 'newspaper',
 'sm': 'print',
 'r': [{'c': [229, 413, 2286, 392],
   'p': [{'c': [229, 413, 2286, 392],
     'l': [{'c': [868, 428, 1050, 30],
       't': [{'c': [868, 433, 41, 25], 'tx': '.Ai'},
        {'c': [1888, 435, 30, 19], 'tx': 'v'}]},
      {'c': [291, 432, 1756, 72],
       't': [{'c': [291, 449, 50, 55], 'tx': '/'},
        {'c': [384, 439, 59, 42], 'tx': '>'},
        {'c': [868, 450, 42, 51], 'tx': 'U'},
        {'c': [1845, 438, 202, 59], 'tx': '/<7'}]},
      {'c': [260, 532, 2164, 43],
       't': [{'c': [260, 542, 77, 33], 'tx': '^Bo'},
        {'c': [397, 541, 58, 33], 'tx': '1H'},
        {'c': [477, 552, 45, 22], 'tx': '™'},
        {'c': [552, 540, 41, 34], 'tx': 'W'},
        {'c': [643, 544, 41, 29], 'tx': 'M'},
        {'c': [714, 540, 46, 33], 'tx': 'M'},
        {'c': [786, 539, 41, 34], 'tx': 'M'},
        {'c': [869, 543, 47, 29], 'tx': 'DM'},
        

In [36]:
# This cell attempts to access the first region
sc_issues[0].pages[0].page_data['r'][0]

{'c': [229, 413, 2286, 392],
 'p': [{'c': [229, 413, 2286, 392],
   'l': [{'c': [868, 428, 1050, 30],
     't': [{'c': [868, 433, 41, 25], 'tx': '.Ai'},
      {'c': [1888, 435, 30, 19], 'tx': 'v'}]},
    {'c': [291, 432, 1756, 72],
     't': [{'c': [291, 449, 50, 55], 'tx': '/'},
      {'c': [384, 439, 59, 42], 'tx': '>'},
      {'c': [868, 450, 42, 51], 'tx': 'U'},
      {'c': [1845, 438, 202, 59], 'tx': '/<7'}]},
    {'c': [260, 532, 2164, 43],
     't': [{'c': [260, 542, 77, 33], 'tx': '^Bo'},
      {'c': [397, 541, 58, 33], 'tx': '1H'},
      {'c': [477, 552, 45, 22], 'tx': '™'},
      {'c': [552, 540, 41, 34], 'tx': 'W'},
      {'c': [643, 544, 41, 29], 'tx': 'M'},
      {'c': [714, 540, 46, 33], 'tx': 'M'},
      {'c': [786, 539, 41, 34], 'tx': 'M'},
      {'c': [869, 543, 47, 29], 'tx': 'DM'},
      {'c': [938, 539, 43, 33], 'tx': 'M'},
      {'c': [1023, 538, 41, 33], 'tx': 'W'},
      {'c': [1094, 538, 41, 33], 'tx': 'W'},
      {'c': [1178, 538, 46, 33], 'tx': 'M'},
      {'c

### Helper Functions for Region Debugging

In [37]:
def find_regions_per_ci(page_regions):
    """Group page regions by content item ID."""
    ci_regions = {}
    unattached_counter = 1
    for region in page_regions:
        if 'pOf' in region:
            og_ci_id = region['pOf']
        else:
            og_ci_id = f"No attached CI {unattached_counter}"
            unattached_counter += 1

        if og_ci_id in ci_regions:
            ci_regions[og_ci_id].append(region)
        else:
            ci_regions[og_ci_id] = [region]
    return ci_regions

In [38]:
colors = ["red", 'green', 'blue', 'purple', 'orange', 'cyan', 'brown', "limegreen", 'pink']

def print_regions_on_page(page_object, issue, color_to_print='all', save=False):
    """Visualize regions and content items on a newspaper page."""
    print(f"Page: {page_object.id}")
    
    # For SUB, we'd need image paths - placeholder for now
    # img_path would come from IIIF or local storage
    print(f"  Number of regions: {len(page_object.page_data.get('r', []))}")
    
    ci_regions = find_regions_per_ci(page_object.page_data['r'])
    
    other_cis_on_page = [ci for ci in issue.issue_data['i'] if page_object.number in ci['m']['pp'] and ci['m']['tp'] == 'image']
    
    print(f"  Content items on page: {len(other_cis_on_page)}")
    for ci_num, ci in enumerate(other_cis_on_page):
        if 'pOf' in ci:
            reg_color = colors[int(ci['pOf'][-4:]) % len(colors)]
            print(f"    {ci['m']['id']}-img_pOf {ci['pOf']} --> CI {reg_color}, coords= {ci.get('c', 'N/A')}")
        else:
            reg_color = colors[ci_num % len(colors)]
            print(f"    {ci['m']['id']} --> additional CI {reg_color}, coords= {ci.get('c', 'N/A')}")
    
    for ci_num, (ci, ci_region_list) in enumerate(ci_regions.items()):
        reg_color = colors[ci_num % len(colors)] if "No attached" in ci else colors[int(ci[-4:]) % len(colors)]
        print(f"  {ci}: {len(ci_region_list)} regions in {reg_color}")

In [39]:
# Run the debug printer on the pages of the first processed issue
for page_object in sc_issues[0].pages:
    print_regions_on_page(page_object, sc_issues[0])

Page: hamb_echo-1888-02-02-a-p0001
  Number of regions: 34
  Content items on page: 1
    hamb_echo-1888-02-02-a-i0009 --> additional CI red, coords= N/A
  hamb_echo-1888-02-02-a-i0008: 2 regions in pink
  hamb_echo-1888-02-02-a-i0007: 11 regions in limegreen
  hamb_echo-1888-02-02-a-i0006: 1 regions in brown
  No attached CI 1: 1 regions in purple
  No attached CI 2: 1 regions in orange
  hamb_echo-1888-02-02-a-i0010: 1 regions in green
  No attached CI 3: 1 regions in brown
  No attached CI 4: 1 regions in limegreen
  No attached CI 5: 1 regions in pink
  No attached CI 6: 1 regions in red
  No attached CI 7: 1 regions in green
  No attached CI 8: 1 regions in blue
  No attached CI 9: 1 regions in purple
  No attached CI 10: 1 regions in orange
  No attached CI 11: 1 regions in cyan
  No attached CI 12: 1 regions in brown
  No attached CI 13: 1 regions in limegreen
  No attached CI 14: 1 regions in pink
  No attached CI 15: 1 regions in red
  No attached CI 16: 1 regions in green
  N

## Summary

This notebook successfully demonstrates:
1. ✓ SUB issue detection and selection
2. ✓ SubNewspaperIssue class initialization and parsing
3. ✓ SubNewspaperPage class functionality
4. ✓ Canonical format output generation
5. ✓ Edition mapping (A1/A2/A3 → a/b/c)
6. ✓ PPN identifier extraction
7. ✓ METS/Alto XML processing

**Observation:** The SUB importer is working correctly in terms of finding issues, parsing METS, and creating issue/page structures. 